
# Lending Game — Part 7: Testing the pricing rules against competitor scenarios

*Part 7 itself (held-out scenarios, regret, bootstrap, final choice) is still to be done.* This first
section only shows how to load the rules tuned in Part 6 (`analysis/pricing_rules.ipynb`), ready to
price any set of applicants.

`load_tuned_rules(train)` reads `data/pricing/tuned_rules.json` and rebuilds every tuned rule as a
function **data → our rate per applicant** (`NaN` = no offer). R4 is rebuilt from its saved decile
edges and margins. R7 refits, on the training set, the model that predicts the competitors' default
probabilities from our own information, exactly as in Part 6. The training set is only needed for
that.

Rules carried to Part 7 (Part 6, section 5): **R4+C** (best one so far?), **R7+C** (also good), **R3**,
**R2**, and **R1** and **B0** as references. The other tuned families (R4, R5, R6, R7 without
cutoff) are loaded too.


In [1]:

import sys
sys.path.append("..")  # make src/ importable from analysis/

import pandas as pd
from IPython.display import display

from src.pricing_rules import load_pricing_data, load_tuned, load_tuned_rules

train = load_pricing_data("train")            # needed to rebuild R7's competitor model
validation = load_pricing_data("validation")  # the data Part 7 scores the rules on

rules = load_tuned_rules(train)               # dict: rule name -> function(data) -> rates
CARRIED = ["R4+C", "R7+C", "R3", "R2", "R1", "B0"]

for name in CARRIED:
    print(f"{name:<5} {rules[name].params}")


R4+C  {'floor': 0.005, 'edges': [0.012217877363987672, 0.027528700365754683, 0.04473841299671522, 0.06449592028448543, 0.0868933784322514, 0.11098440254371388, 0.1327701842927805, 0.15419245650430682, 0.2055395915504812], 'centres': [0.00528012992564965, 0.01966243357343165, 0.0358029324211695, 0.0542164159133189, 0.0753354734398568, 0.09861224861761686, 0.1232345023098893, 0.14090826676618684, 0.1774464593593158, 0.2507941100730716], 'margins': [0.0175, 0.0175, 0.02, 0.03166666666666667, 0.03166666666666667, 0.03166666666666667, 0.16875, 0.16875, 0.19375, 0.19375], 'cutoff': 0.125}
R7+C  {'delta': -0.02, 'q': 0.5, 'a': 0.4, 'floor': 0.005, 'cutoff': 0.1}
R3    {'a': 0.35, 'm': 0.01, 'floor': 0.02}
R2    {'k': 0.7, 'floor': 0.02}
R1    {'m': 0.04, 'floor': 0.005}
B0    {'floor': 0.02}



**Using a rule.** Call it on any data with the columns of `load_pricing_data` (here the validation
set). It returns our rate for each applicant, indexed by `row_id`, with `NaN` where we make no offer.
The result is the `r3` column of the offers table that `simulate_market` takes, next to the
competitors' rates from `make_scenario`.


In [2]:

rates = rules["R4+C"](validation)
print(rates.head())

overview = pd.DataFrame({
    name: {"offers made": rules[name](validation).notna().mean(),
           "median rate offered": rules[name](validation).median(),
           "mean rate offered": rules[name](validation).mean()}
    for name in CARRIED}).T
display(overview.style.format("{:.2%}"))

# The T6 competitor margins (average of the best-response cycle) are saved with the rules.
print("T6 competitor margins:", load_tuned()["equilibrium"]["T6_margins"])


row_id
77965         NaN
75716    0.097236
49786    0.057651
10820    0.166610
81056    0.019238
Name: r3, dtype: float64


,offers made,median rate offered,mean rate offered
R4+C,65.30%,8.14%,9.64%
R7+C,55.25%,6.49%,7.48%
R3,99.83%,14.49%,17.86%
R2,99.56%,16.08%,20.02%
R1,100.00%,13.54%,15.97%
B0,100.00%,9.54%,12.13%


T6 competitor margins: [0.0425, 0.0375]
